# LEAR ensemble: full-run predictions and metrics

Loads the completed backtest over `load-windsolar` features — hour-by-hour ensemble
forecast vs. observed price for every zone, plus the per-zone error summary — and
plots a sample week for DK1.

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# Resolve experiments/ relative to the notebook, wherever it is opened from.
CANDIDATES = [Path("experiments"), Path("../experiments"), Path("../../experiments")]
EXPERIMENTS = next((p for p in CANDIDATES if (p / "predictions_load-windsolar.csv").exists()), None)
if EXPERIMENTS is None:
    raise FileNotFoundError(
        "predictions_load-windsolar.csv not found under experiments/. Set EXPERIMENTS manually."
    )
print("experiments dir:", EXPERIMENTS.resolve())

## 1. Load predictions, observations and metrics

In [ ]:
predictions = pd.read_csv(EXPERIMENTS / "predictions_load-windsolar.csv", parse_dates=["timestamp_local"])
summary = pd.read_csv(EXPERIMENTS / "summary_load-windsolar.csv")

print(f"predictions: {len(predictions):,} rows, zones: {sorted(predictions['zone'].unique())}")
print(f"span: {predictions['timestamp_local'].min()} -> {predictions['timestamp_local'].max()}")
predictions.head(3)

## 2. Per-zone metrics

`mae_ensemble` / `rmae_ensemble` are the ensemble's error; the `mae_cw*` / `rmae_cw*`
columns are the individual calibration-window members it's built from.

In [ ]:
summary.sort_values("mae_ensemble").reset_index(drop=True)

## 3. Sample week: DK1 observed vs. forecast

A full Monday–Sunday week taken from the middle of the run, so the sample avoids both
the unusually calm opening days and the two DST-gap hours flagged as `unobserved_hours`
in the summary above.

In [ ]:
dk1 = predictions[predictions["zone"] == "DK1"].set_index("timestamp_local").sort_index()

mid = dk1.index.min() + (dk1.index.max() - dk1.index.min()) / 2
week_start = (mid - pd.Timedelta(days=mid.dayofweek)).normalize()
week_end = week_start + pd.Timedelta(days=7)
sample = dk1.loc[week_start:week_end - pd.Timedelta(hours=1)]

COLOR_OBSERVED = "#2a78d6"
COLOR_FORECAST = "#eb6834"

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sample.index, sample["observed"], color=COLOR_OBSERVED, linewidth=1.8, label="Observed price")
ax.plot(sample.index, sample["forecast"], color=COLOR_FORECAST, linewidth=1.8, label="Ensemble forecast")

ax.set_title(
    f"DK1 day-ahead price: observed vs. LEAR ensemble forecast\n"
    f"{week_start.date()} – {(week_end - pd.Timedelta(days=1)).date()}"
)
ax.set_ylabel("EUR/MWh")
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
ax.grid(True, color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()